# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 1: verify the modeling setup

import pandas as pd
import numpy as np

# Dataset is already loaded in the notebook as df
print("Dataset loaded successfully.")

# Observed binary label:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Rows:", len(df))
print("Label: is_declining_label = (trend_direction == 'down')")
print("Declining pages:", int(df["is_declining_label"].sum()))
print("Non-declining pages:", int((df["is_declining_label"] == 0).sum()))

print("\nModels:")
print("- Logistic Regression")
print("- Decision Tree")
print("- Random Forest")

print("\nEvaluation metric: Precision@50")
print("Split: client-holdout")
print("Label-derived fields used as features:", [])

Dataset loaded successfully.
Rows: 30000
Label: is_declining_label = (trend_direction == 'down')
Declining pages: 16262
Non-declining pages: 13738

Models:
- Logistic Regression
- Decision Tree
- Random Forest

Evaluation metric: Precision@50
Split: client-holdout
Label-derived fields used as features: []


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 2: create a client-holdout split

from sklearn.model_selection import train_test_split

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_mask = df["client_id"].isin(train_clients)
test_mask = df["client_id"].isin(test_clients)

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("=== CLIENT-HOLDOUT SPLIT ===")
print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nClient overlap:",
      len(set(train_clients) & set(test_clients)))

print("\nTraining label rate:",
      round(train_df["is_declining_label"].mean(), 4))

print("Test label rate:",
      round(test_df["is_declining_label"].mean(), 4))

assert len(set(train_clients) & set(test_clients)) == 0

print("\nResult: No client appears in both train and test.")

=== CLIENT-HOLDOUT SPLIT ===
Total clients: 32
Training clients: 25
Test clients: 7

Training rows: 26581
Test rows: 3419

Client overlap: 0

Training label rate: 0.5444
Test label rate: 0.5238

Result: No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 3: train models and compare against the baseline

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# Feature definitions from the previous assignments
# ---------------------------------------------------------

feature_fields = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
]

# Explicit leakage check.
leakage_fields = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

print("Feature count:", len(feature_fields))
print("Leakage/excluded fields:", leakage_fields)

assert "trend_direction" not in feature_fields
assert "trend_pct" not in feature_fields
assert "content_id" not in feature_fields
assert "client_id" not in feature_fields
assert "provider_used" not in feature_fields
assert "model_used" not in feature_fields

X_train = train_df[feature_fields].copy()
X_test = test_df[feature_fields].copy()

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

categorical_features = [
    c for c in feature_fields
    if train_df[c].dtype == "object"
]

numeric_features = [
    c for c in feature_fields
    if c not in categorical_features
]

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# ---------------------------------------------------------
# Models
# ---------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
}

# ---------------------------------------------------------
# Precision@K
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


results = []
model_predictions = {}

# ---------------------------------------------------------
# Train and evaluate learned models
# ---------------------------------------------------------

for model_name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    probabilities = pipeline.predict_proba(X_test)[:, 1]

    precision_50 = precision_at_k(
        y_test.values,
        probabilities,
        k=50
    )

    model_predictions[model_name] = probabilities

    results.append({
        "Method": model_name,
        "Precision@50": precision_50
    })

# ---------------------------------------------------------
# Recreate Week-4 baseline on TEST SET ONLY
# ---------------------------------------------------------

baseline_test = test_df.copy()

def percentile_rank_against_test(series):
    return series.rank(
        pct=True,
        method="average"
    ).fillna(0)


impressions = baseline_test["impressions_90d"].fillna(
    baseline_test["impressions_90d"].median()
)

days_update = baseline_test["days_since_last_update"].fillna(
    baseline_test["days_since_last_update"].median()
)

avg_position = baseline_test["avg_position"].fillna(
    baseline_test["avg_position"].median()
)

word_count = baseline_test["word_count"].fillna(
    baseline_test["word_count"].median()
)

visibility_score = percentile_rank_against_test(
    np.log1p(impressions)
)

freshness_risk_score = percentile_rank_against_test(
    days_update
)

position_component = (
    1 - percentile_rank_against_test(
        avg_position.clip(lower=1, upper=50)
    )
)

position_opportunity_score = (
    position_component
    * visibility_score
    * (avg_position > 0).astype(int)
)

depth_gap_score = (
    1 - percentile_rank_against_test(word_count)
) * visibility_score

baseline_score = (
    0.40 * visibility_score
    + 0.30 * freshness_risk_score
    + 0.25 * position_opportunity_score
    + 0.05 * depth_gap_score
)

baseline_precision_50 = precision_at_k(
    baseline_test["is_declining_label"].values,
    baseline_score.values,
    k=50
)

results.append({
    "Method": "Week-4 Baseline",
    "Precision@50": baseline_precision_50
})

comparison = (
    pd.DataFrame(results)
    .sort_values(
        "Precision@50",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n=== MODEL VS BASELINE ===")
display(comparison)

print(
    "\nBest learned model:",
    comparison[
        comparison["Method"] != "Week-4 Baseline"
    ].iloc[0]["Method"]
)

print(
    "Best learned Precision@50:",
    round(
        comparison[
            comparison["Method"] != "Week-4 Baseline"
        ].iloc[0]["Precision@50"],
        4
    )
)

print(
    "Baseline Precision@50:",
    round(baseline_precision_50, 4)
)

Feature count: 38
Leakage/excluded fields: ['trend_direction', 'trend_pct', 'content_id', 'client_id', 'provider_used', 'model_used']

Numeric features: 29
Categorical features: 9

=== MODEL VS BASELINE ===


,Method,Precision@50
0,Logistic Regression,1.00
1,Decision Tree,1.00
2,Random Forest,0.98
3,Week-4 Baseline,0.46



Best learned model: Logistic Regression
Best learned Precision@50: 1.0
Baseline Precision@50: 0.46


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 4: error analysis for the best learned model

# Identify best learned model from the comparison table.
learned_results = comparison[
    comparison["Method"] != "Week-4 Baseline"
]

best_model_name = learned_results.iloc[0]["Method"]
best_scores = model_predictions[best_model_name]

error_df = test_df[
    [
        "content_id",
        "client_id",
        "trend_direction",
        "trend_pct",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr"
    ]
].copy()

error_df["model_score"] = best_scores

error_df["predicted_top50"] = False
top50_indices = np.argsort(best_scores)[::-1][:50]

error_df.iloc[top50_indices, error_df.columns.get_loc(
    "predicted_top50"
)] = True

error_df["actual_declining"] = (
    error_df["trend_direction"] == "down"
)

error_df["error_type"] = "Correct / outside top-50"

error_df.loc[
    error_df["predicted_top50"]
    & ~error_df["actual_declining"],
    "error_type"
] = "False positive"

error_df.loc[
    ~error_df["predicted_top50"]
    & error_df["actual_declining"],
    "error_type"
] = "False negative"

print("=== BEST MODEL ===")
print(best_model_name)

print("\n=== ERROR COUNTS ===")
display(
    error_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

print("\n=== TOP MODEL PICKS ===")
display(
    error_df
    .sort_values("model_score", ascending=False)
    .head(20)
)

print("\n=== FALSE POSITIVES IN TOP-50 ===")
display(
    error_df[
        error_df["error_type"] == "False positive"
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
)

print("\n=== MISSED DECLINING PAGES ===")
display(
    error_df[
        error_df["error_type"] == "False negative"
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
)

print("\nInterpretation:")
print(
    "The model captures observed patterns associated with declining pages, "
    "but individual errors remain."
)
print(
    "High model score means higher decision-support priority, "
    "not a causal explanation."
)

=== BEST MODEL ===
Logistic Regression

=== ERROR COUNTS ===


,error_type,count
0,False negative,1741
1,Correct / outside top-50,1678



=== TOP MODEL PICKS ===


,content_id,client_id,trend_direction,trend_pct,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,avg_position,ctr,model_score,predicted_top50,actual_declining,error_type
25056,content_6605878eeb0d,client_a88a7902cb,down,-89.4,10238,25,33,20,11.9,0.24,1.000000,True,True,Correct / outside top-50
12707,content_8e5645f83c08,client_bbb965ab0c,down,-77.6,14866,27,52,8,8.2,0.18,1.000000,True,True,Correct / outside top-50
19198,content_4a519bf8cdbd,client_bbb965ab0c,down,-73.2,12673,30,48,15,5.5,0.24,1.000000,True,True,Correct / outside top-50
22550,content_4351cb73a022,client_bbb965ab0c,down,-48.2,26759,31,61,15,9.0,0.12,1.000000,True,True,Correct / outside top-50
14621,content_0390a273c940,client_a88a7902cb,down,-68.2,10484,36,64,20,13.1,0.34,1.000000,True,True,Correct / outside top-50
8438,content_b5b60616573b,client_bbb965ab0c,down,-71.6,16286,19,89,15,8.0,0.12,1.000000,True,True,Correct / outside top-50
23606,content_0f61f4c94440,client_a88a7902cb,down,-77.1,7534,7,17,28,9.9,0.09,1.000000,True,True,Correct / outside top-50
1502,content_6eeeb0e9f975,client_bbb965ab0c,down,-94.5,3922,1,6,20,4.8,0.03,0.999999,True,True,Correct / outside top-50
9344,content_41dd7803de3b,client_a88a7902cb,down,-57.3,9560,14,37,20,13.0,0.15,0.999998,True,True,Correct / outside top-50
8692,content_ce861b52509e,client_9400f1b21c,down,-57.3,13975,9,7,20,7.3,0.06,0.999998,True,True,Correct / outside top-50



=== FALSE POSITIVES IN TOP-50 ===


,content_id,client_id,trend_direction,trend_pct,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,avg_position,ctr,model_score,predicted_top50,actual_declining,error_type



=== MISSED DECLINING PAGES ===


,content_id,client_id,trend_direction,trend_pct,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,avg_position,ctr,model_score,predicted_top50,actual_declining,error_type
21974,content_3234d64d00b5,client_a88a7902cb,down,-60.1,5109,12,29,20,12.9,0.23,0.996742,False,True,False negative
28031,content_ca122cc888aa,client_bbb965ab0c,down,-88.1,1863,1,2,15,33.2,0.05,0.996709,False,True,False negative
23462,content_e4243546d781,client_9400f1b21c,down,-84.3,2482,5,3,20,8.6,0.20,0.996047,False,True,False negative
27686,content_1123b27c2c89,client_8b940be7fb,down,-39.3,7204,69,40,20,4.0,0.96,0.995652,False,True,False negative
6713,content_aaa7641512c3,client_9400f1b21c,down,-52.2,8807,6,9,20,7.8,0.07,0.995015,False,True,False negative
21430,content_0ca7254d6c74,client_bbb965ab0c,down,-49.9,4920,30,48,15,6.9,0.61,0.994648,False,True,False negative
8107,content_6d57a796c762,client_bbb965ab0c,down,-80.3,3095,2,10,15,13.3,0.06,0.993926,False,True,False negative
17056,content_9ffd07ea9034,client_bbb965ab0c,down,-43.6,7260,37,45,15,6.8,0.51,0.993626,False,True,False negative
29617,content_e014d9a2f1f2,client_a88a7902cb,down,-50.9,5037,5,11,20,26.4,0.10,0.993351,False,True,False negative
27254,content_3c641718b524,client_bbb965ab0c,down,-65.7,4276,14,23,15,10.1,0.33,0.992998,False,True,False negative



Interpretation:
The model captures observed patterns associated with declining pages, but individual errors remain.
High model score means higher decision-support priority, not a causal explanation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.